---

Digital Brain

---

In [1]:
# import pgl commands
from pgl import pgl, pglExperiment, pglTask, pglParameter, pglKeyBuffer, pglEyeTrackingCalibrationTask, pglMessageAckTask, pglMessages, pglMovieDatabase

# import libraries
import json
from pathlib import Path
import numpy as np

# initialize pgl
pgl = pgl()
pgl.cleanUp()

================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl:checkOS) Python version: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 17:06:14) [Clang 19.1.7 ]
(pgl:checkOS) Running on Mac mini (Mac14,3) with macOS version: 15.6.1
(pgl:checkOS) Apple M2 Cores: 8 (4 performance and 4 efficiency) Memory: 16 GB
(pgl:checkOS) GPU: Apple M2 (Built-In) 10 cores, Metal 3 support
(pgl:checkOS)   HP ZR2440w [Main Display]: 1920 x 1200 (WUXGA - Widescreen Ultra eXtended Graphics Array) (Unknown type) GammaTable size: 1024
(pgl:checkOS)   VIEWPixx3D: 1920 x 1080 (1080p FHD - Full High Definition) (Unknown type) GammaTable size: 1024
(pglBase) M

---

Display settings

---

In [ ]:
pgl.displaySettings()

---

Settings

---

In [ ]:
pgl.settings()

---

Eye calibration task

---

In [ ]:
# Set up experiment
e = pglExperiment(pgl, experimentName="Eye calibration")

for iCalibration in range(5):
    # tell the subject what will happen
    messageAckTask = pglMessageAckTask(pgl, "Press a key to do eye calibration")
    messageAckTask.settings.phaseNum=iCalibration*2
    e.addTask(messageAckTask)

    # calibration task
    calibrationTask = pglEyeTrackingCalibrationTask(pgl, nCalibrationPoints=17, calibrationWidth=15, calibrationHeight=15)
    calibrationTask.settings.phaseNum=iCalibration*2+1
    e.addTask(calibrationTask)



---

Run

---

In [ ]:
e.initScreen()
e.run()
e.display()

---

Memory only task

---

In [ ]:
class pglDescriptionTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # initialize the key buffer
        self.keyBuffer = pglKeyBuffer(maxLineLength=40)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Description Task"
        
        # set seglens
        self.settings.seglen = [2, float('inf'), 15]

        # fixed parameters, these will automatically be saved in the settings file
        self.settings.fixedParameters = {
            #'moviePath':'/Users/Shared/digital-assets/stimulus/digital/0008',
            'moviePath':'/Users/justin/Desktop/testvideos',
            'displayWidth': 30,
            
        }        
        p = self.settings.fixedParameters
        
        # load movie database
        self.mdb = pglMovieDatabase(p['moviePath'])
        
        # get movie names
        self.addParameter(pglParameter('movieNum',np.arange(self.mdb.nStimuli)))
                    
    ########################
    def startSegment(self, startTime):
        '''
        Start a segment.
        '''
        super().startSegment(startTime)
        
        if self.state.currentSegment == 0:
            self.e.flush = True
            # do not eat keys
            self.e.setEatAllKeys(False)
            # load the movie
            moviePath = self.mdb.stimuli[self.currentParams['movieNum']].filename
            self.m = pgl.movie(filename=str(moviePath),displayWidth=self.settings.fixedParameters['displayWidth'])
            print(f"m: {self.m} moviePath: {moviePath}")
        elif self.state.currentSegment == 1:
            # play the movie
            self.m.play(displayWidth=self.settings.fixedParameters['displayWidth'])
            self.jumpSegment()

        elif self.state.currentSegment == 2:
            # description segment
            self.e.setEatAllKeys(True)
            self.keyBuffer.clear()
            self.state.keyBufferDirty=False
            self.state.elapsedTime = -1
            self.e.flush = False
        
    ########################
    def updateScreen(self):
        if self.state.currentSegment == 0:
            pass
        elif self.state.currentSegment == 2:
            # calcluate elapsed time
            elapsedTime = round(self.settings.seglen[self.state.currentSegment]-(self.pgl.getSecs()-self.state.segmentStartTime),0)
            
            # decide if we need to draw (only if elapsed time has changed or keyBufferDirty)
            if elapsedTime != self.state.elapsedTime or self.state.keyBufferDirty:
                # update elapsed time
                self.state.elapsedTime = elapsedTime
                # draw text
                self.pgl.text(f"Description Task: {elapsedTime:0.1f}", line="center")
            
                # draw the subject text
                text = self.keyBuffer.getWrappedText()
                for line in text.split('\n'):
                    self.pgl.text(line)
                self.state.keyBufferDirty = False
                
                # flush screen
                self.e.pgl.flush()
    
    ########################
    def handleEvents(self, events):
        for event in events:
            if event.eventType == 'keydown':
                self.keyBuffer.processEvent(event)
                self.state.keyBufferDirty = True


---

Initialize task

---

In [6]:
# Set up experiment
e = pglExperiment(pgl, experimentName="Digital Brain Memory pilot")

# initialize tasks
descriptionTask = pglDescriptionTask(pgl)

# add the tasks to the experiment
e.addTask(descriptionTask)

(pglMovieDatabase->pglStimulusDatabase:__init__) Found 1 stimulus files in testvideos


---

Run task

---

In [7]:
# initialize screen
e.initScreen()

# and run the experiment
e.run()

# display results
e.display()

================================ pglBase:close =================================
(pglBase:close) Closing connection to mglMetal application
(pgl->pglBase:close) Closing mglMetal application with PID 37007
(pgl->pglBase:close) mglMetal application with PID: 37007 was killed successfully.
(pgl:_pglComm) Closed socket: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260823_145339.Ae2rxlIkGq
(pglBase:removeOrphanedSockets) No orphaned sockets found in /Users/justin/Library/Containers/gru.mglMetal/Data
(pgl:_resolution:getResolution) Display 1/2: 1920x1080 60Hz 32bits
(pglExperiment:initScreen) Changing screen resolution to: 1920 x 1200 60Hz 32bits from: 1920 x 1080 60Hz 32bits
================================= pglBase:open =================================
(pgl->pglBase:open) Starting mglMetal application: /Users/justin/proj/pgl/metal/mglMetal.app
(pgl->pglBase:open) Using socket with address: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260823_

1495 thread_policy_set failed: 4.
1495 thread_policy_set failed: 4.
1495 thread_policy_set failed: 4.
1495 thread_policy_set failed: 4.


(pglEyelink:openEDF) Data file opened: 20260823.edf
(pglEyelink:__init__) (pglEyelink) Using pgl display for Eyelink calibration and validation.
(pglEyelink:sendMessage) Sending message pgl: start date=2026/08/23
(pglEyelink:sendMessage) Sending message pgl: start time=14:54:29
(pglEyelink:sendMessage) Sending message pgl: start isoformat=2026-08-23T14:54:29.560219
(pglEyelink:sendMessage) Sending message pgl: start getSecs=9957125.294582585
(pglEyelink:sendMessage) Sending message pgl: start screenWidthPix=1920 screenHeightPix=1080 screenWidthDeg=78.16782546127375
screenHeightDeg=46.88693436427129
(pglEyelink:start) Eye tracking started.
(pglExperiment:calibrateEyeTracker) Press space to calibrate eye tracker. return to skip.
(pglEventListener) Eating 2 keys: ['return', 'space']
(pglExperiment:calibrateEyeTracker) Skipping eye tracker calibration.
(pglEventListener) Eating 7 keys: ['1', '2', '3', '4', '`', 'escape', 'space']
Block 1: 1 trials randomized over: ['movieNum']
(pglStimulus

AttributeError: 'pglStimulusMovie' object has no attribute 'movieNum'

---

Movie database

---

In [ ]:
from pgl import pglMovieDatabase
mdb = pglMovieDatabase('/Users/Shared/digital-assets/stimulus/digital/0008')

In [ ]:
mdb.display(stimulusNum=5)
#pgl.traitsDialog(mdb)

In [ ]:
pgl.traitsDialog(mdb)

In [ ]:
mdb.stimuli[0].filename

In [ ]:
range(100)